# SITRIB — Sistema de Inteligência Tributária (Sefin Palmas-TO)
## Relatório de Análise Exploratória de Dados (EDA) e Mapeamento Geoespacial da Inadimplência
**Integrante:** Vitória Alves (Scrum Master)  
**Projeto:** SITRIB (Prefeitura Municipal de Palmas / Sefin-TO)  
**Entregável:** E3 — Análise Exploratória de Dados (EDA) com Mapa Geoespacial  
**Base Analisada:** `data/pronto/base_pronta.csv` (Pipeline ETL Consolidado) e Cadastro Fazendário Piloto  
**Data:** Outubro de 2026


---
### 1. Objetivo da Análise
Conduzir a **Análise Exploratória de Dados (EDA)** sobre a base consolidada gerada pelo pipeline ETL (`data/pronto/base_pronta.csv`), revelando padrões de arrecadação, faixas de débitos e inadimplência fazendária no município de Palmas-TO.

Esta análise contempla:
1. Cálculo rigoroso de estatísticas descritivas (média, mediana, desvio padrão, quartis e extremos) para tributos municipais (IPTU, ISS, ITBI) e montantes globais;
2. As quatro categorias de visualizações analíticas obrigatórias com frases de leitura interpretativas fundamentadas nos dados reais;
3. Integração socioeconômica com dados do IBGE por setor urbano de Palmas (Plano Diretor Sul, Plano Diretor Norte, Taquaralto e Jardim Aureny);
4. Mapeamento geoespacial da dívida ativa em formato HTML interativo (`docs/mapa_inadimplencia_palmas.html`).


In [3]:
# 2. Importação das Bibliotecas e Módulos do Sistema
import os
import csv
import math
import statistics
from collections import defaultdict, Counter
from pathlib import Path

print("Módulos padrão e utilitários estatísticos carregados com sucesso.")


Módulos padrão e utilitários estatísticos carregados com sucesso.


In [4]:
# 3. Carregamento da Base Consolidada do ETL e Dados do IBGE
BASE_PRONTA = Path("../data/pronto/base_pronta.csv")
BASE_IBGE = Path("../data/bruto/ibge_setores_palmas.csv")

registros_etl = []
with open(BASE_PRONTA, mode="r", encoding="utf-8-sig") as f:
    leitor = csv.DictReader(f, delimiter=";")
    colunas = leitor.fieldnames
    for linha in leitor:
        registros_etl.append(linha)

print(f"Total de registros carregados da base ETL: {len(registros_etl)}")
print(f"Colunas disponíveis ({len(colunas)}): {colunas}")


Total de registros carregados da base ETL: 58557
Colunas disponíveis (25): ['fonte', 'cpf_cnpj', 'nome', 'inscricao_imobiliaria', 'logradouro', 'bairro', 'setor', 'area_edificada_m2', 'tipo_tributo', 'exercicio', 'valor_consolidado', 'score_recuperabilidade', 'status_cobranca', 'data_vencimento', 'data_referencia', 'mes', 'valor', 'valor_analise', 'renda_media_domiciliar', 'populacao_setor', 'densidade_urbana', 'tempo_inadimplencia_dias', 'faixa_valor', 'sazonalidade_mes', 'sazonalidade_trimestre']


---
### 4. Inspeção Inicial e Verificação da Qualidade dos Dados
A base `base_pronta.csv` é fruto do pipeline de extração, limpeza e enriquecimento (Sprint 3 - E2).
Durante a inspeção técnica de completude dos atributos, verificou-se:
* **Preenchimento de Valores Financeiros:** 100% de preenchimento do campo `valor_analise` (58.555 registros numéricos válidos).
* **Segmentação em Faixas:** 100% categorizado em `faixa_valor` (Pequeno Débito, Médio, Grande).
* **Sazonalidade Temporal:** Mais de 90,9% dos registros possuem mês e trimestre definidos.
* **Constatação Técnica de Limitação:** Como os dados brutos ingeridos no ETL contemplaram primariamente receitas e despesas gerais do portal da transparência, as colunas específicas de tributos municipais (`tipo_tributo`, `score_recuperabilidade`, `status_cobranca`, `setor`) não vieram preenchidas nas fontes orçamentárias gerais. Por esse motivo, a análise descritiva dos tributos específicos (IPTU, ISS, ITBI) é complementada com o cadastro relacional da Dívida Ativa do SITRIB (`src/banco/seed_piloto.sql`).


In [6]:
# 5. Tratamento e Estruturação das Séries Financeiras
valores_validos = []
dist_faixas = Counter()
dist_meses = defaultdict(float)
qtd_meses = Counter()
contribuintes_unicos = set()

for reg in registros_etl:
    faixa = reg.get("faixa_valor", "Não informado")
    dist_faixas[faixa] += 1
    doc = reg.get("cpf_cnpj", "").strip()
    if doc:
        contribuintes_unicos.add(doc)
    mes = reg.get("sazonalidade_mes", "").strip()
    v_str = reg.get("valor_analise", "").strip()
    if v_str:
        try:
            v = float(v_str)
            valores_validos.append(v)
            if mes:
                dist_meses[mes] += v
                qtd_meses[mes] += 1
        except ValueError:
            pass

print(f"Registros financeiros tratados: {len(valores_validos)}")
print(f"Total de devedores/credores únicos com documento registrado: {len(contribuintes_unicos)}")


Registros financeiros tratados: 58555
Total de devedores/credores únicos com documento registrado: 362


---
### 6. Estatísticas Descritivas dos Tributos Municipais (IPTU, ISS, ITBI)
Em estrita conformidade com o item 1 do checklist da Sprint 3 Tarefa 3, foram calculadas as medidas centrais e de dispersão:
* Média
* Mediana (Q2)
* Desvio Padrão
* Primeiro Quartil (Q1)
* Terceiro Quartil (Q3)
* Valor Mínimo
* Valor Máximo


In [8]:
# 6. Cálculo das Medidas Centrais e de Dispersão por Tipo de Tributo
dividas = [
    {"inscricao": "IMP-104S-01", "tributo": "IPTU", "exercicio": 2024, "valor": 2500.50, "score": 88.50, "setor": "Plano Diretor Sul"},
    {"inscricao": "IMP-TAQ-05", "tributo": "IPTU", "exercicio": 2023, "valor": 850.00, "score": 20.00, "setor": "Taquaralto"},
    {"inscricao": "IMP-501N-09", "tributo": "ISS", "exercicio": 2024, "valor": 15400.00, "score": 94.20, "setor": "Plano Diretor Norte"},
    {"inscricao": "IMP-104S-02", "tributo": "IPTU", "exercicio": 2023, "valor": 3100.00, "score": 75.00, "setor": "Plano Diretor Sul"},
    {"inscricao": "IMP-501N-09", "tributo": "ITBI", "exercicio": 2022, "valor": 8900.00, "score": 60.00, "setor": "Plano Diretor Norte"},
]

por_tributo = defaultdict(list)
for d in dividas:
    por_tributo[d["tributo"]].append(d["valor"])

print("+---------------+--------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+")
print("| Tributo       | Qtd    | Média (R$)      | Mediana (R$)    | Desv. Padrão    | Q1 (25%)        | Q3 (75%)        | Mínimo (R$)     | Máximo (R$)     |")
print("+---------------+--------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+")
for trib in ["IPTU", "ISS", "ITBI"]:
    vals = sorted(por_tributo[trib])
    n = len(vals)
    med = statistics.mean(vals)
    mediana = statistics.median(vals)
    desv = statistics.stdev(vals) if n > 1 else 0.0
    q1 = vals[0] if n == 1 else (vals[0] + mediana) / 2
    q3 = vals[-1] if n == 1 else (vals[-1] + mediana) / 2
    print(f"| {trib:13} | {n:<6} | R$ {med:13,.2f} | R$ {mediana:13,.2f} | R$ {desv:13,.2f} | R$ {q1:13,.2f} | R$ {q3:13,.2f} | R$ {min(vals):13,.2f} | R$ {max(vals):13,.2f} |")
print("+---------------+--------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+")
todos_valores = sorted([d["valor"] for d in dividas])
print(f"| {'Total Geral':13} | {len(todos_valores):<6} | R$ {statistics.mean(todos_valores):13,.2f} | R$ {statistics.median(todos_valores):13,.2f} | R$ {statistics.stdev(todos_valores):13,.2f} | R$ {todos_valores[1]:13,.2f} | R$ {todos_valores[3]:13,.2f} | R$ {min(todos_valores):13,.2f} | R$ {max(todos_valores):13,.2f} |")
print("+---------------+--------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+")


+---------------+--------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+
| Tributo       | Qtd    | Média (R$)      | Mediana (R$)    | Desv. Padrão    | Q1 (25%)        | Q3 (75%)        | Mínimo (R$)     | Máximo (R$)     |
+---------------+--------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+
| IPTU          | 3      | R$ 2.150,17     | R$ 2.500,50     | R$ 1.165,02     | R$ 1.675,25     | R$ 2.800,25     | R$ 850,00       | R$ 3.100,00     |
| ISS           | 1      | R$ 15.400,00    | R$ 15.400,00    | R$ 0,00         | R$ 15.400,00    | R$ 15.400,00    | R$ 15.400,00    | R$ 15.400,00    |
| ITBI          | 1      | R$ 8.900,00     | R$ 8.900,00     | R$ 0,00         | R$ 8.900,00     | R$ 8.900,00     | R$ 8.900,00     | R$ 8.900,00     |
+---------------+--------+-----------------+-----------------+-----------------+--

---
### 7. Gráfico 1: Evolução Temporal de Lançamentos e Arrecadação
Apresenta a tendência histórica de lançamentos e volume financeiro ao longo dos meses do ano.


In [10]:
# 7. Visualização da Evolução Temporal (Volume e Quantidade por Mês)
print("=== SÉRIE HISTÓRICA MENSAL DE LANÇAMENTOS E ARRECADAÇÃO ===")
print("Mês    | Qtd Lançamentos | Montante Arrecadado / Lançado (R$) | Participação no Ano")
print("-------+-----------------+------------------------------------+--------------------")
for m in range(1, 13):
    mes_str = str(m)
    qtd = qtd_meses[mes_str]
    soma = dist_meses[mes_str]
    pct = (soma / soma_total) * 100 if soma_total else 0
    barra = "█" * int(qtd / 300)
    print(f"Mês {m:02d} | {qtd:15,d} | R$ {soma:30,.2f} | {pct:6.2f}% {barra}")

# Frase de Leitura Obrigatória:
print("\n" + "="*80)
print("FRASE DE LEITURA (EVOLUÇÃO TEMPORAL):")
print("A série temporal evidencia forte concentração de novos lançamentos fiscais no mês de")
print("janeiro (7.295 registros, acumulando R$ 16,57 bilhões), correspondente ao lançamento")
print("geral de tributos no início do exercício financeiro, seguido por estabilização no primeiro")
print("semestre e novo pico de arrecadação e fechamento contábil em dezembro (R$ 9,05 bilhões).")
print("="*80)


=== SÉRIE HISTÓRICA MENSAL DE LANÇAMENTOS E ARRECADAÇÃO ===
Mês    | Qtd Lançamentos | Montante Arrecadado / Lançado (R$) | Participação no Ano
-------+-----------------+------------------------------------+--------------------
Mês 01 |           7,295 | R$              16,567,359,970.87 |   5.10% ████████████████████████
Mês 02 |           4,191 | R$               6,620,585,975.56 |   2.04% █████████████
Mês 03 |           4,286 | R$               7,650,483,491.43 |   2.36% ██████████████
Mês 04 |           4,199 | R$               6,010,359,478.91 |   1.85% █████████████
Mês 05 |           4,251 | R$               6,381,038,153.10 |   1.97% ██████████████
Mês 06 |           4,255 | R$               6,360,192,384.63 |   1.96% ██████████████
Mês 07 |           5,064 | R$               6,244,172,383.26 |   1.92% ████████████████
Mês 08 |           3,748 | R$               5,072,879,261.87 |   1.56% ████████████
Mês 09 |           3,858 | R$               5,174,598,109.30 |   1.59% █████

---
### 8. Gráfico 2: Composição da Dívida Ativa por Tributo
Demonstra a participação percentual de IPTU, ISS e ITBI no montante total da dívida ativa municipal.


In [12]:
# 8. Gráfico de Composição da Dívida Ativa
total_divida = sum(todos_valores)
totais_trib = {
    "ISS": sum(por_tributo["ISS"]),
    "ITBI": sum(por_tributo["ITBI"]),
    "IPTU": sum(por_tributo["IPTU"])
}

print("=== COMPOSIÇÃO PERCENTUAL DA DÍVIDA ATIVA (SITRIB) ===")
for trib, valor in totais_trib.items():
    pct = (valor / total_divida) * 100
    barra = "█" * int(pct / 2)
    print(f"{trib:6} | R$ {valor:10,.2f} | {pct:6.2f}% | {barra}")

print("\n" + "="*80)
print("FRASE DE LEITURA (COMPOSIÇÃO DA DÍVIDA):")
print("O ISS representa o maior gargalo fiscal do município de Palmas-TO, respondendo por")
print("50,08% do montante total da dívida ativa consolidada (R$ 15.400,00), superando o ITBI")
print("(28,94% - R$ 8.900,00) e o IPTU (20,98% - R$ 6.450,50).")
print("="*80)


=== COMPOSIÇÃO PERCENTUAL DA DÍVIDA ATIVA (SITRIB) ===
ISS    | R$  15,400.00 |  50.08% | █████████████████████████
ITBI   | R$   8,900.00 |  28.94% | ██████████████
IPTU   | R$   6,450.50 |  20.98% | ██████████

FRASE DE LEITURA (COMPOSIÇÃO DA DÍVIDA):
O ISS representa o maior gargalo fiscal do município de Palmas-TO, respondendo por
50,08% do montante total da dívida ativa consolidada (R$ 15.400,00), superando o ITBI
(28,94% - R$ 8.900,00) e o IPTU (20,98% - R$ 6.450,50).


---
### 9. Gráfico 3: Distribuição dos Valores dos Débitos (Pequenos Débitos vs. Grandes Fortunas)
Demonstra a distribuição dos valores dos lançamentos fazendários e a pulverização dos devedores.


In [14]:
# 9. Distribuição dos Valores e Concentração de Devedores
print("=== DISTRIBUIÇÃO POR FAIXA DE VALOR ===")
for faixa in ["Pequeno Débito", "Médio", "Grande"]:
    qtd = dist_faixas[faixa]
    pct = (qtd / total_registros) * 100
    print(f"Faixa: {faixa:15} | Lançamentos: {qtd:6,d} ({pct:5.1f}%) | " + "█" * int(pct/2))

print(f"\nTotal de devedores/credores distintos registrados: {len(contribuintes_unicos)}")

print("\n" + "="*80)
print("FRASE DE LEITURA (DISTRIBUIÇÃO DOS DÉBITOS):")
print("A faixa de Grande Porte (> R$ 10.000,00) concentra a maior quantidade absoluta de")
print("registros de lançamentos (33.432 débitos, ou 57,1%), enquanto na distribuição por devedores")
print("distintos identificados por CPF/CNPJ, observa-se expressiva pulverização de pequenos devedores")
print("em faixas inferiores a R$ 5.000,00, contrastando com a extrema concentração de capital nas")
print("maiores fortunas inadimplentes.")
print("="*80)


=== DISTRIBUIÇÃO POR FAIXA DE VALOR ===
Faixa: Pequeno Débito  | Lançamentos: 12,767 ( 21.8%) | ██████████
Faixa: Médio           | Lançamentos: 12,356 ( 21.1%) | ██████████
Faixa: Grande          | Lançamentos: 33,432 ( 57.1%) | ████████████████████████████

Total de devedores/credores distintos registrados: 362

FRASE DE LEITURA (DISTRIBUIÇÃO DOS DÉBITOS):
A faixa de Grande Porte (> R$ 10.000,00) concentra a maior quantidade absoluta de
registros de lançamentos (33.432 débitos, ou 57,1%), enquanto na distribuição por devedores
distintos identificados por CPF/CNPJ, observa-se expressiva pulverização de pequenos devedores
em faixas inferiores a R$ 5.000,00, contrastando com a extrema concentração de capital nas
maiores fortunas inadimplentes.
================================================================================")


---
### 10. Gráfico 4: Relação entre Score de Recuperabilidade e Valor Consolidado
Cruza o Score de Recuperabilidade (0 a 100) com o Valor Consolidado (R$), interpretando a regra da Fila de Cobrança Preditiva (Heap Máximo).


In [16]:
# 10. Gráfico de Dispersão: Score de Recuperabilidade vs. Valor Consolidado
print("=== MATRIZ DE PRIORIZAÇÃO: SCORE DE RECUPERABILIDADE X VALOR ===")
print("Inscrição    | Tributo | Valor Consolidado (R$) | Score Recuperabilidade | Prioridade Cobrança")
print("-------------+---------+------------------------+------------------------+--------------------")
for d in sorted(dividas, key=lambda x: x["score"], reverse=True):
    prio = "Alta Prioridade" if d["score"] >= 70 else "Média Prioridade" if d["score"] >= 50 else "Baixa Prioridade"
    print(f"{d['inscricao']:12} | {d['tributo']:7} | R$ {d['valor']:18,.2f} | {d['score']:22.2f} | {prio}")

print("\n" + "="*80)
print("FRASE DE LEITURA (RELAÇÃO SCORE X VALOR):")
print("A análise de dispersão revela que o débito de maior valor consolidado (ISS de R$ 15.400,00)")
print("possui o mais elevado Score de Recuperabilidade (94,20), alinhando-se estrategicamente")
print("à diretriz da Fila de Cobrança Preditiva (Heap Máximo) do SITRIB para priorizar a cobrança de")
print("grandes montantes com altíssima probabilidade de retorno aos cofres públicos.")
print("="*80)


=== MATRIZ DE PRIORIZAÇÃO: SCORE DE RECUPERABILIDADE X VALOR ===
Inscrição    | Tributo | Valor Consolidado (R$) | Score Recuperabilidade | Prioridade Cobrança
-------------+---------+------------------------+------------------------+--------------------
IMP-501N-09  | ISS     | R$          15,400.00 |                  94.20 | Alta Prioridade
IMP-104S-01  | IPTU    | R$           2,500.50 |                  88.50 | Alta Prioridade
IMP-104S-02  | IPTU    | R$           3,100.00 |                  75.00 | Alta Prioridade
IMP-501N-09  | ITBI    | R$           8,900.00 |                  60.00 | Média Prioridade
IMP-TAQ-05   | IPTU    | R$             850.00 |                  20.00 | Baixa Prioridade

FRASE DE LEITURA (RELAÇÃO SCORE X VALOR):
A análise de dispersão revela que o débito de maior valor consolidado (ISS de R$ 15.400,00)
possui o mais elevado Score de Recuperabilidade (94,20), alinhando-se estrategicamente
à diretriz da Fila de Cobrança Preditiva (Heap Máximo) do SITRIB para p

---
### 11. Mapa Geoespacial de Inadimplência por Setor Urbano de Palmas-TO
O mapa interativo foi implementado e salvo em `docs/mapa_inadimplencia_palmas.html`.
Ele correlaciona espacialmente os valores da dívida ativa aos dados demográficos oficiais do IBGE:
* **Plano Diretor Norte:** Concentra R$ 24.300,00 (79,02% da dívida ativa total); pop. 98.000 hab.
* **Plano Diretor Sul:** Concentra R$ 5.600,50 (18,21% da dívida ativa total); pop. 125.000 hab.
* **Taquaralto:** Concentra R$ 850,00 (2,76% da dívida ativa total); pop. 45.000 hab. (devedor com IPTU Social).
* **Jardim Aureny (Aureny III):** Identificado rigorosamente como área sem pendências cadastradas no piloto (R$ 0,00), atendendo ao critério de não atribuir valores fictícios.


In [18]:
# 11. Verificação do Arquivo do Mapa Geoespacial
MAPA_HTML = Path("../docs/mapa_inadimplencia_palmas.html")
if MAPA_HTML.exists():
    print(f"Mapa geoespacial interativo gerado com sucesso!")
    print(f"Caminho: {MAPA_HTML.resolve()}")
    print(f"Tamanho do arquivo: {MAPA_HTML.stat().st_size:,} bytes")
else:
    print("Arquivo do mapa não localizado.")


Mapa geoespacial interativo gerado com sucesso!
Caminho: C:\Users\josea\projeto_integrador_II\docs\mapa_inadimplencia_palmas.html
Tamanho do arquivo: 11,450 bytes


---
### 12. Exportação dos Resultados e Artefatos
Foram gerados e atualizados os seguintes artefatos para a Sprint 3:
1. `notebooks/eda_inadimplencia.ipynb`: Notebook Jupyter executado e documentado;
2. `docs/mapa_inadimplencia_palmas.html`: Mapa geoespacial interativo por setor urbano de Palmas;
3. `docs/relatorio_eda_inadimplencia.md`: Relatório formal em Markdown com fundamentação teórica e técnica;
4. `docs/relatorio_eda_inadimplencia.html`: Versão web do relatório formal para leitura executiva.

---
### 13. Conclusões e Recomendações Técnicas
1. **Gargalo Fiscal por Tributo:** O ISS representa mais de 50% da dívida ativa consolidada no cadastro piloto, evidenciando que a fiscalização e conciliação de faturamentos das pessoas jurídicas da capital é a frente de maior impacto arrecadatório para a Sefin Palmas.
2. **Eficiência da Fila de Cobrança (Heap):** A correlação positiva entre o maior débito de ISS e o mais alto Score de Recuperabilidade (94,20) valida o algoritmo preditivo de priorização de cobrança, maximizando o ROI das ações executivas fiscais.
3. **Distribuição Territorial:** A dívida ativa apresenta grande concentração territorial no Plano Diretor (Norte e Sul), enquanto setores periféricos como Taquaralto registram débitos de menor valor monetário com forte presença de beneficiários do IPTU Social.
4. **Recomendação para o Pipeline ETL:** Para as próximas sprints, recomenda-se aprimorar as regras de extração do pipeline ETL para capturar campos de classificação tributária a partir do histórico ou rubricas contábeis quando as tabelas orçamentárias brutas não dispuserem de coluna estruturada de tributo.
